# Quick guide

This script processes spectra stored in a **multi-sheet Excel workbook** and fits fringe peaks with a **two-stage Gaussian workflow**.

For each sheet, it:
- reads wavelength and intensity data
- converts wavelength (nm) to wavenumber (µm⁻¹)
- subtracts the baseline
- smooths the spectrum with Savitzky-Golay
- detects candidate peaks
- fits Gaussians on the smoothed spectrum (**Stage 1**)
- refits accepted peaks on the raw baseline-subtracted spectrum (**Stage 2**)
- removes overlapping fits using a spacing filter
- calculates delta values from the fitted fringe centres
- saves peak tables, delta summaries, and overview plots

Fringe numbers are assigned once after initial peak selection and are **not renumbered later**.

## Input format

Each worksheet must use this fixed layout:
- **column 0** = wavelength in **nm**
- **column 1** = intensity
- data start at **row index 2** (the first two rows are skipped)

## How to run

1. Set the file paths at the top of the script:
   - `INPUT_WORKBOOK`
   - `OUTPUT_DIR`

2. Check the analysis settings in `FringeConfig`.

3. Run the script in Python.  
   It will process **all sheets** in the workbook.

## What you may need to change for your own data

### File locations
Edit:
- `INPUT_WORKBOOK`
- `OUTPUT_DIR`

### Main fitting and detection settings
These are the most likely parameters to adjust:

- `avg_peak_spacing_um_inv`  
  approximate fringe spacing; affects fitting window size

- `peak_prominence`  
  increase if too many weak peaks are detected; decrease if real peaks are missed

- `min_peak_spacing_um_inv`  
  minimum allowed spacing between neighbouring peaks

- `r2_min_processed`  
  Stage 1 fit-quality threshold 

- `r2_min_raw`  
  Stage 2 fit-quality threshold

- `width_min_um_inv`, `width_max_um_inv`  
  allowed Gaussian width range

- `sg_window`, `sg_poly`  
  smoothing strength for the Savitzky-Golay filter

### Baseline settings
If the background shape changes significantly between datasets, you may also need to adjust:
- `baseline_lambda`
- `baseline_p`

## Output

For each sheet, the script writes:
- `<sheet>_Peaks.xlsx`  
  with `Raw` and `Processed` peak tables
- `<sheet>_Delta.xlsx`  
  with per-sheet delta values
- `<sheet>_Overview.png`
- `<sheet>_Overview_Processed.png`

It also writes:
- `Delta_Summary.xlsx`  
  containing one summary row per sheet

## Notes

- Always check the overview plots when using a new dataset.
- If fitting fails for a sheet, the script still writes empty output files and records the sheet status.
- The fixed worksheet layout is assumed throughout this version of the script.

In [4]:
from __future__ import annotations

"""Fringe fitting workflow for spectra stored in a multi-sheet Excel workbook.

How to use
----------
1. Set INPUT_WORKBOOK and OUTPUT_DIR.
2. Keep the Excel layout fixed:
   - column 0 = wavelength (nm)
   - column 1 = intensity
   - data start at row index 2
3. Run the script.

Outputs
-------
For each sheet, the script writes:
- <sheet>_Peaks.xlsx with Raw and Processed sheets
- <sheet>_Delta.xlsx
- overview plot(s)

It also writes a global Delta_Summary.xlsx file.

Fringe numbering
----------------
Fringe numbers are assigned once after initial peak selection and are never
renumbered later. If a fringe is removed by fit-quality or spacing filters,
the remaining fringes keep their original numbers.
"""

import logging
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import curve_fit
from scipy.signal import find_peaks, savgol_filter
from scipy.sparse.linalg import spsolve


# -----------------------------------------------------------------------------
# User file locations
# Edit these for your own dataset.
# -----------------------------------------------------------------------------
INPUT_WORKBOOK = Path("example_data/Fringes.xlsx")
OUTPUT_DIR = Path("example_output")
SUMMARY_WORKBOOK = OUTPUT_DIR / "Delta_Summary.xlsx"


# -----------------------------------------------------------------------------
# Analysis settings
# Edit these if your spectra need different detection or fitting behaviour.
# -----------------------------------------------------------------------------
@dataclass(frozen=True)
class FringeConfig:
    avg_peak_spacing_um_inv: float = 0.004   # approximate fringe spacing
    baseline_lambda: float = 1e7             # ALS baseline smoothness
    baseline_p: float = 0.1                  # ALS asymmetry
    baseline_iterations: int = 500

    sg_window: int = 17                      # Savitzky-Golay window length
    sg_poly: int = 2                         # Savitzky-Golay polynomial order

    peak_prominence: float = 40.0            # peak detection strength threshold
    peak_height: float = 0.0
    peak_distance_samples: int = 1
    min_peak_spacing_um_inv: float = 0.0028  # minimum allowed fitted peak spacing

    r2_min_processed: float = 0.5            # Stage 1 minimum R²
    r2_min_raw: float = 0.2                  # Stage 2 minimum R²

    amp_max_factor: float = 1.2
    width_min_um_inv: float = 0.00045
    width_max_um_inv: float = 0.002
    fit_half_window_factor: float = 0.6

    delta_target_min_pairs: int = 20
    delta_max_relative_deviation: float = 0.015
    delta_spacing_scan: tuple[int, ...] = (25, 20, 15, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1)

    plot_dpi: int = 100


CONFIG = FringeConfig()
FWHM_TO_SIGMA = 1 / (2 * np.sqrt(2 * np.log(2)))

plt.rcParams.update({"figure.dpi": CONFIG.plot_dpi})

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("fringe")


# -----------------------------------------------------------------------------
# Basic helpers
# -----------------------------------------------------------------------------
def wavelength_to_wavenumber(wavelength_nm: np.ndarray) -> np.ndarray:
    return 1000.0 / wavelength_nm


def gaussian(x: np.ndarray, amplitude: float, centre: float, width: float) -> np.ndarray:
    return amplitude * np.exp(-((x - centre) ** 2) / (2 * width ** 2))


def shift_positive(y: np.ndarray) -> np.ndarray:
    y_shifted = y.copy()
    if y_shifted.min() < 0:
        y_shifted += abs(0.05 * y_shifted.min())
    return y_shifted


def als_baseline(y: np.ndarray, lam: float, p: float, niter: int) -> np.ndarray:
    n = len(y)
    d = sparse.diags([1, -2, 1], [0, -1, -2], shape=(n, n - 2))
    w = np.ones(n)

    for _ in range(niter):
        z = spsolve(sparse.spdiags(w, 0, n, n) + lam * d @ d.T, w * y)
        w = p * (y > z) + (1 - p) * (y < z)

    return z


def safe_savgol(y: np.ndarray, desired_window: int, polyorder: int) -> tuple[np.ndarray, int | None]:
    """Apply Savitzky-Golay safely for any spectrum length."""
    n = len(y)
    if n < 3 or n <= polyorder:
        return y.copy(), None

    max_window = n if n % 2 == 1 else n - 1
    window = min(desired_window, max_window)
    if window % 2 == 0:
        window -= 1

    min_window = polyorder + 1
    if min_window % 2 == 0:
        min_window += 1

    if window < min_window:
        window = min_window

    if window > max_window:
        return y.copy(), None

    return savgol_filter(y, window, polyorder), window


def fit_r2(y_true: np.ndarray, y_fit: np.ndarray) -> float:
    residuals = y_true - y_fit
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0


# -----------------------------------------------------------------------------
# Output helpers
# -----------------------------------------------------------------------------
def make_summary_row(
    sheet_name: str,
    status: str,
    avg_delta_um: float = np.nan,
    std_delta_um: float = np.nan,
    median_delta_um: float = np.nan,
    num_pairs: int = 0,
    avg_delta_proc_um: float = np.nan,
    std_delta_proc_um: float = np.nan,
    median_delta_proc_um: float = np.nan,
    num_pairs_proc: int = 0,
) -> dict:
    return {
        "Sheet": sheet_name,
        "Status": status,
        "Average Delta (µm)": avg_delta_um,
        "Std Delta (µm)": std_delta_um,
        "Median Delta (µm)": median_delta_um,
        "Num Pairs Used": num_pairs,
        "Average Delta Proc (µm)": avg_delta_proc_um,
        "Std Delta Proc (µm)": std_delta_proc_um,
        "Median Delta Proc (µm)": median_delta_proc_um,
        "Num Pairs Used Proc": num_pairs_proc,
    }


def empty_raw_table() -> pd.DataFrame:
    return pd.DataFrame(
        columns=[
            "Fringe #",
            "Centre (µm⁻¹)",
            "Wavelength (nm)",
            "Amplitude",
            "Width (µm⁻¹)",
            "R² (Proc)",
            "R² (Raw)",
        ]
    )


def empty_processed_table() -> pd.DataFrame:
    return pd.DataFrame(
        columns=[
            "Fringe #",
            "Centre (µm⁻¹)",
            "Wavelength (nm)",
            "Amplitude",
            "Width (µm⁻¹)",
            "R² (Proc)",
        ]
    )


def write_peak_workbook(
    sheet_output_dir: Path,
    sheet_name: str,
    raw_df: pd.DataFrame,
    processed_df: pd.DataFrame,
) -> None:
    with pd.ExcelWriter(sheet_output_dir / f"{sheet_name}_Peaks.xlsx", engine="xlsxwriter") as writer:
        raw_df.to_excel(writer, sheet_name="Raw", index=False)
        processed_df.to_excel(writer, sheet_name="Processed", index=False)


def write_delta_workbook(
    sheet_output_dir: Path,
    sheet_name: str,
    status: str,
    avg_delta_um_raw: float,
    std_delta_um_raw: float,
    median_delta_um_raw: float,
    num_pairs_raw: int,
    avg_delta_um_proc: float,
    std_delta_um_proc: float,
    median_delta_um_proc: float,
    num_pairs_proc: int,
) -> None:
    pd.DataFrame(
        {
            "Status": [status],
            "Average Delta Raw (µm)": [avg_delta_um_raw],
            "Std Delta Raw (µm)": [std_delta_um_raw],
            "Median Delta Raw (µm)": [median_delta_um_raw],
            "Num Pairs Used Raw": [num_pairs_raw],
            "Average Delta Proc (µm)": [avg_delta_um_proc],
            "Std Delta Proc (µm)": [std_delta_um_proc],
            "Median Delta Proc (µm)": [median_delta_um_proc],
            "Num Pairs Used Proc": [num_pairs_proc],
        }
    ).to_excel(sheet_output_dir / f"{sheet_name}_Delta.xlsx", index=False)


def write_empty_outputs(sheet_name: str, status: str) -> dict:
    sheet_output_dir = OUTPUT_DIR / sheet_name
    sheet_output_dir.mkdir(parents=True, exist_ok=True)

    write_peak_workbook(sheet_output_dir, sheet_name, empty_raw_table(), empty_processed_table())
    write_delta_workbook(
        sheet_output_dir,
        sheet_name,
        status=status,
        avg_delta_um_raw=np.nan,
        std_delta_um_raw=np.nan,
        median_delta_um_raw=np.nan,
        num_pairs_raw=0,
        avg_delta_um_proc=np.nan,
        std_delta_um_proc=np.nan,
        median_delta_um_proc=np.nan,
        num_pairs_proc=0,
    )
    return make_summary_row(sheet_name=sheet_name, status=status)


# -----------------------------------------------------------------------------
# Core analysis helpers
# -----------------------------------------------------------------------------
def load_sheet_data(sheet_name: str) -> tuple[np.ndarray, np.ndarray]:
    """Load one worksheet using the fixed input layout.

    Expected layout:
    - column 0 = wavelength (nm)
    - column 1 = intensity
    - data start at row index 2
    """
    sheet_df = pd.read_excel(INPUT_WORKBOOK, sheet_name=sheet_name, header=None)
    wavelength_nm = pd.to_numeric(sheet_df.iloc[2:, 0], errors="coerce").to_numpy(float)
    intensity_raw = pd.to_numeric(sheet_df.iloc[2:, 1], errors="coerce").to_numpy(float)

    wavenumber_um_inv = wavelength_to_wavenumber(wavelength_nm)
    sort_order = np.argsort(wavenumber_um_inv)
    return wavenumber_um_inv[sort_order], intensity_raw[sort_order]


def preprocess_spectrum(
    wavenumber_um_inv: np.ndarray,
    intensity_raw: np.ndarray,
    config: FringeConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    baseline = als_baseline(
        y=intensity_raw,
        lam=config.baseline_lambda,
        p=config.baseline_p,
        niter=config.baseline_iterations,
    )

    intensity_corrected = shift_positive(intensity_raw - baseline)
    intensity_smoothed, sg_window_used = safe_savgol(
        intensity_corrected,
        desired_window=config.sg_window,
        polyorder=config.sg_poly,
    )

    if sg_window_used is None:
        log.warning(
            "  ▶ Savitzky-Golay skipped: spectrum too short for window=%d, poly=%d",
            config.sg_window,
            config.sg_poly,
        )
    elif sg_window_used != config.sg_window:
        log.info("  ▶ Savitzky-Golay window adjusted to %d", sg_window_used)

    intensity_raw_bs = shift_positive(intensity_raw - baseline)
    return wavenumber_um_inv, intensity_raw_bs, intensity_smoothed


def detect_initial_peaks(
    wavenumber_um_inv: np.ndarray,
    intensity_smoothed: np.ndarray,
    config: FringeConfig,
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    peak_indices, peak_properties = find_peaks(
        intensity_smoothed,
        prominence=config.peak_prominence,
        distance=config.peak_distance_samples,
        height=config.peak_height,
    )

    log.info("  ▶ find_peaks proposed %d candidates", len(peak_indices))
    if len(peak_indices) == 0:
        return peak_indices, peak_properties

    prominences = peak_properties["prominences"]
    strongest_first = np.argsort(prominences)[::-1]
    kept_positions = []

    for pos in strongest_first:
        peak_idx = peak_indices[pos]
        if not kept_positions or np.all(
            np.abs(wavenumber_um_inv[peak_idx] - wavenumber_um_inv[peak_indices[kept_positions]])
            >= config.min_peak_spacing_um_inv
        ):
            kept_positions.append(pos)

    peak_indices = peak_indices[kept_positions]
    peak_properties = {k: v[kept_positions] for k, v in peak_properties.items()}

    log.info(
        "  ▶ after Δν ≥ %.4f filter, %d remain",
        config.min_peak_spacing_um_inv,
        len(peak_indices),
    )

    descending_wavenumber = np.argsort(wavenumber_um_inv[peak_indices])[::-1]
    peak_indices = peak_indices[descending_wavenumber]
    peak_properties = {k: v[descending_wavenumber] for k, v in peak_properties.items()}
    return peak_indices, peak_properties


def make_fit_results_array(n_fringes: int) -> np.ndarray:
    dtype = np.dtype(
        [("amp", "f8"), ("cen", "f8"), ("wid", "f8"), ("r2_proc", "f8"), ("r2_raw", "f8")]
    )
    return np.full(n_fringes, np.nan, dtype=dtype)


def stage1_fit_processed(
    wavenumber_um_inv: np.ndarray,
    intensity_smoothed: np.ndarray,
    peak_indices: np.ndarray,
    config: FringeConfig,
) -> np.ndarray:
    fit_results = make_fit_results_array(len(peak_indices))
    initial_amplitudes = intensity_smoothed[peak_indices]
    initial_centres = wavenumber_um_inv[peak_indices]
    fit_half_window = config.fit_half_window_factor * (config.avg_peak_spacing_um_inv / 2)

    for fringe_idx, (amplitude_guess, centre_guess) in enumerate(zip(initial_amplitudes, initial_centres)):
        local_mask = (
            (wavenumber_um_inv >= centre_guess - fit_half_window)
            & (wavenumber_um_inv <= centre_guess + fit_half_window)
        )
        if local_mask.sum() < 3:
            continue

        width_guess = max(config.avg_peak_spacing_um_inv * FWHM_TO_SIGMA, config.width_min_um_inv * 1.1)
        width_guess = min(width_guess, config.width_max_um_inv * 0.9)

        bounds = (
            [0, centre_guess - fit_half_window, config.width_min_um_inv],
            [config.amp_max_factor * intensity_smoothed.max(), centre_guess + fit_half_window, config.width_max_um_inv],
        )

        try:
            amplitude_fit, centre_fit, width_fit = curve_fit(
                gaussian,
                wavenumber_um_inv[local_mask],
                intensity_smoothed[local_mask],
                p0=[amplitude_guess, centre_guess, width_guess],
                bounds=bounds,
                maxfev=12_000,
            )[0]
        except (ValueError, RuntimeError):
            continue

        y_fit = gaussian(wavenumber_um_inv[local_mask], amplitude_fit, centre_fit, width_fit)
        r2_processed = fit_r2(intensity_smoothed[local_mask], y_fit)
        if r2_processed < config.r2_min_processed:
            continue

        fit_results[fringe_idx]["amp"] = amplitude_fit
        fit_results[fringe_idx]["cen"] = centre_fit
        fit_results[fringe_idx]["wid"] = width_fit
        fit_results[fringe_idx]["r2_proc"] = r2_processed

    return fit_results


def stage2_fit_raw(
    wavenumber_um_inv: np.ndarray,
    intensity_raw_bs: np.ndarray,
    stage1_results: np.ndarray,
    config: FringeConfig,
) -> np.ndarray:
    fit_results = stage1_results.copy()
    fit_half_window = config.fit_half_window_factor * (config.avg_peak_spacing_um_inv / 2)

    for fringe_idx, record in enumerate(fit_results):
        if np.isnan(record["r2_proc"]):
            continue

        amplitude_guess = max(record["amp"], 1e-3)
        centre_guess = record["cen"]
        width_guess = np.clip(record["wid"], config.width_min_um_inv, config.width_max_um_inv)

        local_mask = (
            (wavenumber_um_inv >= centre_guess - fit_half_window)
            & (wavenumber_um_inv <= centre_guess + fit_half_window)
        )
        local_x = wavenumber_um_inv[local_mask]
        local_y = intensity_raw_bs[local_mask]
        if local_x.size < 3:
            fit_results[fringe_idx]["r2_raw"] = np.nan
            continue

        lower_bounds = [0.5 * amplitude_guess, centre_guess - fit_half_window, config.width_min_um_inv]
        upper_bounds = [2.0 * amplitude_guess, centre_guess + fit_half_window, config.width_max_um_inv]
        if lower_bounds[1] >= upper_bounds[1] or lower_bounds[2] >= upper_bounds[2]:
            fit_results[fringe_idx]["r2_raw"] = np.nan
            continue

        try:
            amplitude_fit, centre_fit, width_fit = curve_fit(
                gaussian,
                local_x,
                local_y,
                p0=[amplitude_guess, centre_guess, width_guess],
                bounds=(lower_bounds, upper_bounds),
                maxfev=20_000,
            )[0]
        except (ValueError, RuntimeError):
            fit_results[fringe_idx]["r2_raw"] = np.nan
            continue

        y_fit = gaussian(local_x, amplitude_fit, centre_fit, width_fit)
        r2_raw = fit_r2(local_y, y_fit)
        if r2_raw < config.r2_min_raw:
            fit_results[fringe_idx]["r2_raw"] = np.nan
            continue

        fit_results[fringe_idx]["amp"] = amplitude_fit
        fit_results[fringe_idx]["cen"] = centre_fit
        fit_results[fringe_idx]["wid"] = width_fit
        fit_results[fringe_idx]["r2_raw"] = r2_raw

    return fit_results


def apply_postfit_spacing_filter(
    fit_results: np.ndarray,
    stage2_mask: np.ndarray,
    min_peak_spacing_um_inv: float,
) -> np.ndarray:
    """Keep the best non-overlapping fits without changing fringe numbers."""
    final_mask = np.zeros_like(stage2_mask, dtype=bool)
    candidate_indices = np.where(stage2_mask)[0]
    if candidate_indices.size == 0:
        return final_mask

    ranked_indices = sorted(
        candidate_indices,
        key=lambda idx: (fit_results["r2_raw"][idx], fit_results["amp"][idx], fit_results["r2_proc"][idx], -idx),
        reverse=True,
    )

    kept_indices: list[int] = []
    for candidate_idx in ranked_indices:
        candidate_centre = fit_results["cen"][candidate_idx]
        if all(
            abs(candidate_centre - fit_results["cen"][kept_idx]) >= min_peak_spacing_um_inv
            for kept_idx in kept_indices
        ):
            kept_indices.append(candidate_idx)
            final_mask[candidate_idx] = True

    return final_mask


def compute_delta_stats(
    centres_um_inv: np.ndarray,
    selected_peak_mask: np.ndarray,
    fringe_numbers: np.ndarray,
    min_peak_spacing_um_inv: float,
    min_fringe_separation: int,
) -> tuple[float, float, float, int]:
    selected_indices = np.where(selected_peak_mask)[0]
    if selected_indices.size < 2:
        return np.nan, np.nan, np.nan, 0

    selected_fringes = fringe_numbers[selected_indices]
    selected_centres = centres_um_inv[selected_indices]
    selected_wavelengths_nm = 1000.0 / selected_centres
    delta_values_nm: list[float] = []

    n_selected = selected_indices.size
    for left_pos in range(n_selected):
        for right_pos in range(left_pos + 1, n_selected):
            pair_order = selected_fringes[right_pos] - selected_fringes[left_pos] + 1
            if pair_order < min_fringe_separation:
                continue

            segment_indices = selected_indices[left_pos:right_pos + 1]
            segment_spacings = np.abs(np.diff(centres_um_inv[segment_indices]))
            if segment_spacings.size == 0 or np.any(segment_spacings < min_peak_spacing_um_inv):
                continue

            lambda_left_nm = selected_wavelengths_nm[left_pos]
            lambda_right_nm = selected_wavelengths_nm[right_pos]
            if np.isclose(lambda_left_nm, lambda_right_nm):
                continue

            delta_nm = (lambda_left_nm * lambda_right_nm * (pair_order - 1)) / (2.0 * (lambda_right_nm - lambda_left_nm))
            delta_values_nm.append(delta_nm)

    if not delta_values_nm:
        return np.nan, np.nan, np.nan, 0

    delta_values_nm = np.asarray(delta_values_nm, dtype=float)
    return (
        float(np.mean(delta_values_nm)) * 0.001,
        float(np.std(delta_values_nm, ddof=0)) * 0.001,
        float(np.median(delta_values_nm)) * 0.001,
        int(delta_values_nm.size),
    )


def select_best_delta(
    label: str,
    centres_um_inv: np.ndarray,
    selected_peak_mask: np.ndarray,
    fringe_numbers: np.ndarray,
    config: FringeConfig,
) -> tuple[float, float, float, int]:
    chosen = None
    best_with_target_pairs = None
    best_overall = None

    for min_fringe_separation in config.delta_spacing_scan:
        avg_delta_um, std_delta_um, median_delta_um, num_pairs = compute_delta_stats(
            centres_um_inv=centres_um_inv,
            selected_peak_mask=selected_peak_mask,
            fringe_numbers=fringe_numbers,
            min_peak_spacing_um_inv=config.min_peak_spacing_um_inv,
            min_fringe_separation=min_fringe_separation,
        )

        if num_pairs == 0 or np.isnan(avg_delta_um) or avg_delta_um == 0:
            continue

        rel_dev = std_delta_um / avg_delta_um
        result = {
            "min_fringe_separation": min_fringe_separation,
            "avg": avg_delta_um,
            "std": std_delta_um,
            "median": median_delta_um,
            "pairs": num_pairs,
            "rel_dev": rel_dev,
        }

        if best_overall is None or rel_dev < best_overall["rel_dev"]:
            best_overall = result

        if num_pairs >= config.delta_target_min_pairs:
            if best_with_target_pairs is None or rel_dev < best_with_target_pairs["rel_dev"]:
                best_with_target_pairs = result

        if num_pairs >= config.delta_target_min_pairs and rel_dev <= config.delta_max_relative_deviation:
            chosen = result
            break

    if chosen is not None:
        log.info(
            "  ▶ delta (%s): min_fringe_sep = %d, %d pairs, rel. dev = %.3f%%",
            label,
            chosen["min_fringe_separation"],
            chosen["pairs"],
            100.0 * chosen["rel_dev"],
        )
        return chosen["avg"], chosen["std"], chosen["median"], chosen["pairs"]

    if best_with_target_pairs is not None:
        log.warning(
            "  ▶ delta (%s): could not reach rel. dev ≤ %.1f%%; using best case with ≥%d pairs: "
            "min_fringe_sep = %d, %d pairs, rel. dev = %.3f%%",
            label,
            100.0 * config.delta_max_relative_deviation,
            config.delta_target_min_pairs,
            best_with_target_pairs["min_fringe_separation"],
            best_with_target_pairs["pairs"],
            100.0 * best_with_target_pairs["rel_dev"],
        )
        return (
            best_with_target_pairs["avg"],
            best_with_target_pairs["std"],
            best_with_target_pairs["median"],
            best_with_target_pairs["pairs"],
        )

    if best_overall is not None:
        log.warning(
            "  ▶ delta (%s): could not obtain %d pairs; falling back to best overall: "
            "min_fringe_sep = %d, %d pairs, rel. dev = %.3f%%",
            label,
            config.delta_target_min_pairs,
            best_overall["min_fringe_separation"],
            best_overall["pairs"],
            100.0 * best_overall["rel_dev"],
        )
        return best_overall["avg"], best_overall["std"], best_overall["median"], best_overall["pairs"]

    log.warning("  ▶ delta (%s): no valid fringe pairs for delta calculation", label)
    return np.nan, np.nan, np.nan, 0


# -----------------------------------------------------------------------------
# Table and plot helpers
# -----------------------------------------------------------------------------
def build_raw_peak_table(fringe_numbers: np.ndarray, fit_results: np.ndarray, final_mask: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "Fringe #": fringe_numbers[final_mask],
            "Centre (µm⁻¹)": fit_results["cen"][final_mask],
            "Wavelength (nm)": 1000.0 / fit_results["cen"][final_mask],
            "Amplitude": fit_results["amp"][final_mask],
            "Width (µm⁻¹)": fit_results["wid"][final_mask],
            "R² (Proc)": fit_results["r2_proc"][final_mask],
            "R² (Raw)": fit_results["r2_raw"][final_mask],
        }
    )


def build_processed_peak_table(
    fringe_numbers: np.ndarray,
    processed_results: np.ndarray,
    processed_mask: np.ndarray,
) -> pd.DataFrame:
    if not np.any(processed_mask):
        return empty_processed_table()

    return pd.DataFrame(
        {
            "Fringe #": fringe_numbers[processed_mask],
            "Centre (µm⁻¹)": processed_results["cen"][processed_mask],
            "Wavelength (nm)": 1000.0 / processed_results["cen"][processed_mask],
            "Amplitude": processed_results["amp"][processed_mask],
            "Width (µm⁻¹)": processed_results["wid"][processed_mask],
            "R² (Proc)": processed_results["r2_proc"][processed_mask],
        }
    )


def plot_raw_overview(
    sheet_output_dir: Path,
    sheet_name: str,
    wavenumber_um_inv: np.ndarray,
    intensity_raw_bs: np.ndarray,
    fringe_numbers: np.ndarray,
    fit_results: np.ndarray,
    final_mask: np.ndarray,
    config: FringeConfig,
) -> None:
    cmap = plt.cm.viridis
    fig, ax = plt.subplots(figsize=(18, 4))
    ax.plot(wavenumber_um_inv, intensity_raw_bs, lw=0.4, c="#333", label="Raw (baseline-subtracted)")

    for record, fringe_number in zip(fit_results[final_mask], fringe_numbers[final_mask]):
        x_fit = np.linspace(record["cen"] - 3 * record["wid"], record["cen"] + 3 * record["wid"], 120)
        ax.plot(x_fit, gaussian(x_fit, record["amp"], record["cen"], record["wid"]), c=cmap(fringe_number / fringe_numbers.max()), lw=1)
        y_peak = gaussian(record["cen"], record["amp"], record["cen"], record["wid"])
        ax.text(record["cen"], y_peak * 1.05, str(fringe_number), ha="center", va="bottom", fontsize=7)

    ax.scatter(
        fit_results["cen"][final_mask],
        gaussian(
            fit_results["cen"][final_mask],
            fit_results["amp"][final_mask],
            fit_results["cen"][final_mask],
            fit_results["wid"][final_mask],
        ),
        c=cmap(fringe_numbers[final_mask] / fringe_numbers.max()),
        s=12,
        zorder=10,
    )
    ax.set_ylim(bottom=-200)
    ax.set_xlabel("ν (µm⁻¹)")
    ax.set_ylabel("I")
    ax.invert_xaxis()
    ax.legend()
    fig.tight_layout()
    fig.savefig(sheet_output_dir / f"{sheet_name}_Overview.png", dpi=config.plot_dpi)
    plt.close(fig)


def plot_processed_overview(
    sheet_output_dir: Path,
    sheet_name: str,
    wavenumber_um_inv: np.ndarray,
    intensity_raw_bs: np.ndarray,
    fringe_numbers: np.ndarray,
    processed_results: np.ndarray,
    processed_mask: np.ndarray,
    config: FringeConfig,
) -> None:
    if not np.any(processed_mask):
        log.warning("%s → no processed peaks available for processed overview plot", sheet_name)
        return

    centres = processed_results["cen"][processed_mask]
    amplitudes = processed_results["amp"][processed_mask]
    widths = processed_results["wid"][processed_mask]
    fringe_subset = fringe_numbers[processed_mask]

    cmap = plt.cm.plasma
    fig, ax = plt.subplots(figsize=(18, 4))
    ax.plot(wavenumber_um_inv, intensity_raw_bs, lw=0.4, c="#333", label="Raw (baseline-subtracted)")

    for centre, amplitude, width, fringe_number in zip(centres, amplitudes, widths, fringe_subset):
        x_fit = np.linspace(centre - 3 * width, centre + 3 * width, 120)
        ax.plot(x_fit, gaussian(x_fit, amplitude, centre, width), c=cmap(fringe_number / fringe_numbers.max()), lw=1)
        y_peak = gaussian(centre, amplitude, centre, width)
        ax.text(centre, y_peak * 1.05, str(fringe_number), ha="center", va="bottom", fontsize=7)

    ax.scatter(
        centres,
        gaussian(centres, amplitudes, centres, widths),
        c=cmap(fringe_subset / fringe_numbers.max()),
        s=12,
        zorder=10,
        label="Processed fits (Stage-1)",
    )
    ax.set_ylim(bottom=-200)
    ax.set_xlabel("ν (µm⁻¹)")
    ax.set_ylabel("I")
    ax.invert_xaxis()
    ax.legend()
    fig.tight_layout()
    fig.savefig(sheet_output_dir / f"{sheet_name}_Overview_Processed.png", dpi=config.plot_dpi)
    plt.close(fig)


# -----------------------------------------------------------------------------
# Per-sheet workflow
# -----------------------------------------------------------------------------
def process_sheet(sheet_name: str, config: FringeConfig) -> dict:
    try:
        log.info("⏳ %s", sheet_name)

        wavenumber_um_inv, intensity_raw = load_sheet_data(sheet_name)
        wavenumber_um_inv, intensity_raw_bs, intensity_smoothed = preprocess_spectrum(
            wavenumber_um_inv,
            intensity_raw,
            config,
        )

        peak_indices, _ = detect_initial_peaks(wavenumber_um_inv, intensity_smoothed, config)
        if len(peak_indices) == 0:
            status = "No peaks detected by find_peaks"
            log.warning("%s → %s", sheet_name, status)
            return write_empty_outputs(sheet_name, status)

        fringe_numbers = np.arange(1, len(peak_indices) + 1)

        stage1_results = stage1_fit_processed(
            wavenumber_um_inv,
            intensity_smoothed,
            peak_indices,
            config,
        )
        processed_mask = ~np.isnan(stage1_results["r2_proc"])
        passed_stage1 = int(np.sum(processed_mask))
        log.info(
            "  ▶ %d/%d passed Stage 1 (R² ≥ %.2f)",
            passed_stage1,
            len(peak_indices),
            config.r2_min_processed,
        )
        if passed_stage1 == 0:
            status = "No peaks passed Stage 1"
            log.warning("%s → %s", sheet_name, status)
            return write_empty_outputs(sheet_name, status)

        processed_results = stage1_results.copy()
        final_results = stage2_fit_raw(
            wavenumber_um_inv,
            intensity_raw_bs,
            stage1_results,
            config,
        )
        stage2_mask = ~np.isnan(final_results["r2_raw"])
        passed_stage2 = int(np.sum(stage2_mask))
        log.info(
            "  ▶ %d/%d passed Stage 2 (R² ≥ %.2f)",
            passed_stage2,
            passed_stage1,
            config.r2_min_raw,
        )
        if passed_stage2 == 0:
            status = "No peaks passed Stage 2"
            log.warning("%s → %s", sheet_name, status)
            return write_empty_outputs(sheet_name, status)

        final_mask = apply_postfit_spacing_filter(
            fit_results=final_results,
            stage2_mask=stage2_mask,
            min_peak_spacing_um_inv=config.min_peak_spacing_um_inv,
        )
        final_count = int(np.sum(final_mask))
        log.info(
            "  ▶ %d peaks kept after post-fit spacing filter (Δν ≥ %.4f)",
            final_count,
            config.min_peak_spacing_um_inv,
        )
        if final_count == 0:
            status = "No peaks remained after post-fit spacing filter"
            log.warning("%s → %s", sheet_name, status)
            return write_empty_outputs(sheet_name, status)

        avg_delta_um_raw, std_delta_um_raw, median_delta_um_raw, num_pairs_raw = select_best_delta(
            label="raw",
            centres_um_inv=final_results["cen"],
            selected_peak_mask=final_mask,
            fringe_numbers=fringe_numbers,
            config=config,
        )
        avg_delta_um_proc, std_delta_um_proc, median_delta_um_proc, num_pairs_proc = select_best_delta(
            label="proc",
            centres_um_inv=processed_results["cen"],
            selected_peak_mask=processed_mask,
            fringe_numbers=fringe_numbers,
            config=config,
        )

        sheet_output_dir = OUTPUT_DIR / sheet_name
        sheet_output_dir.mkdir(parents=True, exist_ok=True)

        raw_peak_df = build_raw_peak_table(fringe_numbers, final_results, final_mask)
        processed_peak_df = build_processed_peak_table(fringe_numbers, processed_results, processed_mask)
        write_peak_workbook(sheet_output_dir, sheet_name, raw_peak_df, processed_peak_df)
        write_delta_workbook(
            sheet_output_dir,
            sheet_name,
            status="OK",
            avg_delta_um_raw=avg_delta_um_raw,
            std_delta_um_raw=std_delta_um_raw,
            median_delta_um_raw=median_delta_um_raw,
            num_pairs_raw=num_pairs_raw,
            avg_delta_um_proc=avg_delta_um_proc,
            std_delta_um_proc=std_delta_um_proc,
            median_delta_um_proc=median_delta_um_proc,
            num_pairs_proc=num_pairs_proc,
        )

        plot_raw_overview(
            sheet_output_dir,
            sheet_name,
            wavenumber_um_inv,
            intensity_raw_bs,
            fringe_numbers,
            final_results,
            final_mask,
            config,
        )
        plot_processed_overview(
            sheet_output_dir,
            sheet_name,
            wavenumber_um_inv,
            intensity_raw_bs,
            fringe_numbers,
            processed_results,
            processed_mask,
            config,
        )

        log.info("%s → OK (%d peaks)", sheet_name, final_count)
        return make_summary_row(
            sheet_name=sheet_name,
            status="OK",
            avg_delta_um=avg_delta_um_raw,
            std_delta_um=std_delta_um_raw,
            median_delta_um=median_delta_um_raw,
            num_pairs=num_pairs_raw,
            avg_delta_proc_um=avg_delta_um_proc,
            std_delta_proc_um=std_delta_um_proc,
            median_delta_proc_um=median_delta_um_proc,
            num_pairs_proc=num_pairs_proc,
        )

    except Exception as exc:
        status = f"Sheet crashed: {exc}"
        log.exception("%s crashed", sheet_name)
        return write_empty_outputs(sheet_name, status)


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main() -> None:
    sheet_names = pd.ExcelFile(INPUT_WORKBOOK).sheet_names
    log.info("%d worksheets detected", len(sheet_names))
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    summary_rows = [process_sheet(sheet_name, CONFIG) for sheet_name in sheet_names]

    if summary_rows:
        pd.DataFrame(summary_rows).to_excel(SUMMARY_WORKBOOK, index=False)
        log.info("💾 Global delta summary saved to %s", SUMMARY_WORKBOOK)
    else:
        log.warning("No delta statistics collected; global summary not written.")

    log.info("🎉 Done")


if __name__ == "__main__":
    main()


16:07:58 [INFO] 82 worksheets detected
16:07:58 [INFO] ⏳ fringes #0.1b_Data
16:08:01 [INFO]   ▶ find_peaks proposed 114 candidates
16:08:01 [INFO]   ▶ after Δν ≥ 0.0028 filter, 39 remain
16:08:01 [INFO]   ▶ 39/39 passed Stage 1 (R² ≥ 0.50)
16:08:01 [INFO]   ▶ 13/39 passed Stage 2 (R² ≥ 0.20)
16:08:01 [INFO]   ▶ 13 peaks kept after post-fit spacing filter (Δν ≥ 0.0028)
16:08:01 [INFO]   ▶ delta (raw): min_fringe_sep = 25, 27 pairs, rel. dev = 0.319%
16:08:01 [INFO]   ▶ delta (proc): min_fringe_sep = 25, 120 pairs, rel. dev = 0.272%
16:08:03 [INFO] fringes #0.1b_Data → OK (13 peaks)
16:08:03 [INFO] ⏳ Fringes #0.1_Data
16:08:07 [INFO]   ▶ find_peaks proposed 95 candidates
16:08:07 [INFO]   ▶ after Δν ≥ 0.0028 filter, 39 remain
16:08:07 [INFO]   ▶ 39/39 passed Stage 1 (R² ≥ 0.50)
16:08:07 [INFO]   ▶ 14/39 passed Stage 2 (R² ≥ 0.20)
16:08:07 [INFO]   ▶ 14 peaks kept after post-fit spacing filter (Δν ≥ 0.0028)
16:08:07 [INFO]   ▶ delta (raw): min_fringe_sep = 15, 31 pairs, rel. dev = 0.579%
